# Model Training

This notebook is optimized for the final deadline sprint.

It does four things:

1. builds a strict grouped evaluation pipeline using the existing `Region` column,
2. tests a small shortlist of target-specific candidates,
3. freezes a safe manifest from full grouped CV,
4. writes three submission files:
   - **A** = safe anchor,
   - **B** = EC aggressive + DRP safe,
   - **C** = hedge blend.

Notes:
- We assume `Region` already exists in the provided dataset.
- We keep the notebook cell-by-cell and avoid one giant integrated script.
- We clip predictions to nonnegative values before submission.

In [1]:
import os
import sys
import json
import time
import hashlib
from datetime import datetime

import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans

from xgboost import XGBRegressor
from IPython.display import display

## Environment and MLflow

In [2]:
sys.path.append(os.path.abspath('..'))

ENV = 'local'   # switch to 'snowflake' if needed

if ENV == 'local':
    from src import config_local as config
else:
    from src import config_snowflake as config

mlflow.set_tracking_uri(config.MLFLOW_URI)
mlflow.set_experiment('WaterQuality')

print('MLflow URI:', config.MLFLOW_URI)

2026/03/12 19:28:47 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/03/12 19:28:47 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/03/12 19:28:47 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/03/12 19:28:47 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/03/12 19:28:47 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/03/12 19:28:47 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/03/12 19:28:48 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/12 19:28:48 INFO alembic.runtime.migration: Will assume non-transactional DDL.


MLflow URI: sqlite:///../mlflow.db


## Global config

This cell defines:
- targets,
- split metadata,
- artifact directory,
- hashing helpers,
- submission integrity checks.

In [5]:
TARGET_COLS = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus'
]

SPLIT_STRATEGY = 'SpatialGroupKFold+PseudoHoldoutGroups'
GROUP_DEFINITION_VERSION = 'kmeans_latlon_v2_group_holdout'
PIPELINE_VERSION = 'deadline_v2_mvp4_spatial_group_holdout'
PREPROCESS_VERSION = 'median_scaler'
ARTIFACT_DIR = '../models/final_deadline_mvp4'

SPATIAL_N_CLUSTERS = 16
CV_N_SPLITS = 5
HOLDOUT_MARGIN_DEG = 0.35
HOLDOUT_MIN_GROUPS = 3
HOLDOUT_MIN_FRAC = 0.08
HOLDOUT_MAX_FRAC = 0.15

os.makedirs(ARTIFACT_DIR, exist_ok=True)


def hash_str(s: str) -> str:
    '''
    Create a short stable hash from a string.
    '''
    return hashlib.sha256(s.encode('utf-8')).hexdigest()[:16]


def hash_list(values) -> str:
    '''
    Hash a list of values after converting to strings.
    '''
    return hash_str('||'.join(map(str, values)))


def compute_group_values_hash(groups: pd.Series) -> str:
    '''
    Hash the exact ordered group assignments.
    Useful to ensure runs are truly comparable.
    '''
    return hash_list(groups.fillna('NA').astype(str).tolist())


def compute_feature_set_hash(features: list) -> str:
    '''
    Hash a feature list in sorted form.
    '''
    return hash_list(sorted(features))


def target_key(target_name: str) -> str:
    '''
    Make a target name filename-safe.
    '''
    return target_name.replace(' ', '')


def make_row_id_template(template_df: pd.DataFrame) -> pd.DataFrame:
    '''
    Add an immutable row_id to the submission template
    so row order can be validated before saving.
    '''
    out = template_df.copy()
    out['row_id'] = np.arange(len(out), dtype=int)
    return out


def assert_submission_integrity(sub_df: pd.DataFrame, template_df: pd.DataFrame, target_cols: list):
    '''
    Validate that the submission is structurally safe.
    '''
    if len(sub_df) != len(template_df):
        raise RuntimeError(f'Row count mismatch: sub={len(sub_df)} template={len(template_df)}')

    if 'row_id' not in sub_df.columns or 'row_id' not in template_df.columns:
        raise RuntimeError('row_id missing in submission/template.')

    if sub_df['row_id'].duplicated().any():
        raise RuntimeError('Duplicate row_id in submission.')

    if not sub_df['row_id'].equals(template_df['row_id']):
        raise RuntimeError('row_id order mismatch.')

    if sub_df[target_cols].isnull().any().any():
        raise RuntimeError('NaN found in target predictions.')

    if (sub_df[target_cols] < 0).any().any():
        raise RuntimeError('Negative predictions found.')


print('Global config loaded.')

Global config loaded.


## Data loading

We load the training data and confirm that the `Region` column already exists.

In [6]:
TRAIN_PATH = '../data/interim/water_quality_mvp_baseline.parquet'
VALID_PATH = '../data/interim/water_quality_mvp_validation.parquet'


df = pd.read_parquet(TRAIN_PATH).copy()
df_val_geo = pd.read_parquet(VALID_PATH)[['Latitude', 'Longitude', 'Sample Date']].copy()

df['Sample Date'] = pd.to_datetime(df['Sample Date'], errors='coerce')
df_val_geo['Sample Date'] = pd.to_datetime(df_val_geo['Sample Date'], errors='coerce')

required_cols = ['Latitude', 'Longitude', 'Sample Date'] + TARGET_COLS
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise RuntimeError(f'Missing required training columns: {missing_cols}')


def add_spatial_groups(data: pd.DataFrame, n_clusters: int = SPATIAL_N_CLUSTERS) -> pd.DataFrame:
    '''
    Create stable spatial groups from latitude/longitude using KMeans.
    '''
    out = data.copy()
    n_clusters = min(max(4, int(n_clusters)), len(out))

    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
    out['spatial_group'] = km.fit_predict(out[['Latitude', 'Longitude']].astype(float)).astype(str)
    return out


def select_pseudo_holdout_groups(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    min_groups: int = HOLDOUT_MIN_GROUPS,
    min_frac: float = HOLDOUT_MIN_FRAC,
    max_frac: float = HOLDOUT_MAX_FRAC,
    margin_deg: float = HOLDOUT_MARGIN_DEG,
):
    '''
    Select whole spatial groups nearest to the validation footprint.
    The selection targets a row fraction range and enforces a minimum number of groups.
    '''
    gdf = train_df.groupby('spatial_group', as_index=False).agg(
        Latitude=('Latitude', 'mean'),
        Longitude=('Longitude', 'mean'),
        n=('spatial_group', 'size')
    )

    lat_min = float(valid_df['Latitude'].min()) - margin_deg
    lat_max = float(valid_df['Latitude'].max()) + margin_deg
    lon_min = float(valid_df['Longitude'].min()) - margin_deg
    lon_max = float(valid_df['Longitude'].max()) + margin_deg

    valid_center_lat = float(valid_df['Latitude'].mean())
    valid_center_lon = float(valid_df['Longitude'].mean())

    lat = gdf['Latitude'].astype(float)
    lon = gdf['Longitude'].astype(float)

    lat_gap = np.maximum(np.maximum(lat_min - lat, 0.0), lat - lat_max)
    lon_gap = np.maximum(np.maximum(lon_min - lon, 0.0), lon - lon_max)

    gdf['bbox_dist'] = np.sqrt(lat_gap ** 2 + lon_gap ** 2)
    gdf['center_dist'] = np.sqrt((lat - valid_center_lat) ** 2 + (lon - valid_center_lon) ** 2)

    gdf = gdf.sort_values(['bbox_dist', 'center_dist', 'n'], ascending=[True, True, False]).reset_index(drop=True)

    total_rows = int(len(train_df))
    selected = []
    selected_rows = 0

    for _, row in gdf.iterrows():
        group_name = str(row['spatial_group'])
        group_rows = int(row['n'])

        need_groups = len(selected) < int(min_groups)
        need_rows = (selected_rows / total_rows) < float(min_frac)

        if need_groups or need_rows:
            selected.append(group_name)
            selected_rows += group_rows
            continue

        next_frac = (selected_rows + group_rows) / total_rows
        if next_frac <= float(max_frac):
            selected.append(group_name)
            selected_rows += group_rows
        else:
            break

    i = len(selected)
    while (len(selected) < int(min_groups) or (selected_rows / total_rows) < float(min_frac)) and i < len(gdf):
        group_name = str(gdf.loc[i, 'spatial_group'])
        if group_name not in selected:
            selected.append(group_name)
            selected_rows += int(gdf.loc[i, 'n'])
        i += 1

    all_groups = set(train_df['spatial_group'].astype(str).unique().tolist())
    if len(selected) >= len(all_groups):
        selected = selected[:-1]

    selected = sorted(set(selected), key=lambda x: int(x))

    return selected


df = add_spatial_groups(df, n_clusters=SPATIAL_N_CLUSTERS)
selected_holdout_groups = select_pseudo_holdout_groups(
    df,
    df_val_geo,
    min_groups=HOLDOUT_MIN_GROUPS,
    min_frac=HOLDOUT_MIN_FRAC,
    max_frac=HOLDOUT_MAX_FRAC,
    margin_deg=HOLDOUT_MARGIN_DEG,
)

df['is_pseudo_valid'] = df['spatial_group'].astype(str).isin(selected_holdout_groups)

print('Training shape:', df.shape)
print('Spatial groups:', df['spatial_group'].nunique())
print('Selected holdout groups:', selected_holdout_groups)
print('Pseudo-validation rows:', int(df['is_pseudo_valid'].sum()))
print('Pseudo-validation share:', round(float(df['is_pseudo_valid'].mean()), 4))
print('\nPseudo-validation by group:')
print(df.groupby('spatial_group')['is_pseudo_valid'].sum().sort_values(ascending=False).head(16))

Training shape: (9319, 12)
Spatial groups: 16
Selected holdout groups: ['3', '9', '10']
Pseudo-validation rows: 1433
Pseudo-validation share: 0.1538

Pseudo-validation by group:
spatial_group
3     509
10    477
9     447
0       0
12      0
13      0
1       0
11      0
15      0
14      0
4       0
2       0
5       0
6       0
7       0
8       0
Name: is_pseudo_valid, dtype: int64


In [7]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9319 entries, 0 to 9318
Data columns (total 12 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Latitude                       9319 non-null   float64       
 1   Longitude                      9319 non-null   float64       
 2   Sample Date                    9319 non-null   datetime64[ns]
 3   swir22                         8234 non-null   float64       
 4   NDMI                           8234 non-null   float64       
 5   MNDWI                          8234 non-null   float64       
 6   pet                            9319 non-null   float64       
 7   Total Alkalinity               9319 non-null   float64       
 8   Electrical Conductance         9319 non-null   float64       
 9   Dissolved Reactive Phosphorus  9319 non-null   float64       
 10  spatial_group                  9319 non-null   object        
 11  is_pseudo_valid  

## Feature engineering

This notebook only uses the two engineered features that were explicitly confirmed from the winning setup:

- `pop_density_upstream`
- `specific_discharge`

In [8]:
def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    '''
    Keep this as a pass-through for MVP 4-feature experiments.
    '''
    return data.copy()


df = engineer_features(df)
print('Feature engineering step completed (no-op for MVP 4-feature setup).')

Feature engineering step completed (no-op for MVP 4-feature setup).


## Frozen feature sets

We use two frozen feature sets:

- **A** = stable base set
- **B** = base + empirical interactions

In [9]:
# Benchmark-like local signal set only
BENCHMARK_4 = ['swir22', 'NDMI', 'MNDWI', 'pet']

FEATURE_SETS = {
    'C': BENCHMARK_4,
}


def features_for_set(df_local: pd.DataFrame, fs_name: str):
    '''
    Return a strict feature list for a named feature set.
    Fail fast if expected columns are missing.
    '''
    requested = FEATURE_SETS[fs_name]
    missing = [f for f in requested if f not in df_local.columns]
    if missing:
        raise RuntimeError(f'Missing in {fs_name}: {missing}')
    return requested


print('Feature sets ready:')
for k, v in FEATURE_SETS.items():
    print(f'{k}: {len(v)} features -> {v}')

Feature sets ready:
C: 4 features -> ['swir22', 'NDMI', 'MNDWI', 'pet']


## Preprocessing

We use median imputation and standard scaling inside the CV pipeline.

In [10]:
def get_preprocessor(features_used):
    '''
    Build the preprocessing pipeline for numeric features.
    '''
    numeric_pipe = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    return ColumnTransformer(
        transformers=[('num', numeric_pipe, features_used)],
        remainder='drop'
    )

## Models and shortlist

This shortlist is deliberately small:
- TA: mostly stable XGB
- EC: linear empirical + one XGB challenger
- DRP: safer linear options + one shallow XGB challenger

In [11]:
from sklearn.ensemble import RandomForestRegressor


def log_wrap(model):
    '''
    Wrap a regressor with log1p / expm1 target transformation.
    '''
    return TransformedTargetRegressor(
        regressor=model,
        func=np.log1p,
        inverse_func=np.expm1
    )


# Centralized defaults (easy to tune in one place)
DEFAULT_XGB_PARAMS = {
    'objective': 'reg:squarederror',
    'n_estimators': 300,
    'learning_rate': 0.03,
    'max_depth': 4,
    'min_child_weight': 10,
    'subsample': 0.70,
    'colsample_bytree': 0.70,
    'reg_alpha': 1.0,
    'reg_lambda': 6.0,
    'random_state': 42,
    'n_jobs': -1,
}

DEFAULT_RF_PARAMS = {
    'n_estimators': 600,
    'min_samples_leaf': 3,
    'max_features': 'sqrt',
    'random_state': 42,
    'n_jobs': -1,
}

MODEL_SPECS = {
    'Ridge_a10_Log': {'kind': 'ridge', 'params': {'alpha': 10.0}, 'log_target': True},
    'Ridge_a30_Log': {'kind': 'ridge', 'params': {'alpha': 30.0}, 'log_target': True},
    'Lasso_a0.0005_Log': {'kind': 'lasso', 'params': {'alpha': 0.0005, 'max_iter': 20000}, 'log_target': True},
    'Lasso_a0.0010_Log': {'kind': 'lasso', 'params': {'alpha': 0.0010, 'max_iter': 20000}, 'log_target': True},
    'Elastic_a0.001_l07_Log': {'kind': 'elastic', 'params': {'alpha': 0.001, 'l1_ratio': 0.7, 'max_iter': 20000, 'random_state': 42}, 'log_target': True},
    'XGB_d4_lr003_Log': {'kind': 'xgb', 'params': {'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 4}, 'log_target': True},
    'XGB_d3_lr003_Log': {'kind': 'xgb', 'params': {'n_estimators': 220, 'learning_rate': 0.03, 'max_depth': 3}, 'log_target': True},
    'RF_n600_raw': {'kind': 'rf', 'params': {}, 'log_target': False},
    'RF_n600_Log': {'kind': 'rf', 'params': {}, 'log_target': True},
}


def build_model_from_spec(spec):
    kind = spec['kind']
    params = spec.get('params', {})
    use_log = spec.get('log_target', True)

    if kind == 'ridge':
        base = Ridge(**params)
    elif kind == 'lasso':
        base = Lasso(**params)
    elif kind == 'elastic':
        base = ElasticNet(**params)
    elif kind == 'xgb':
        p = DEFAULT_XGB_PARAMS.copy()
        p.update(params)
        base = XGBRegressor(**p)
    elif kind == 'rf':
        p = DEFAULT_RF_PARAMS.copy()
        p.update(params)
        base = RandomForestRegressor(**p)
    else:
        raise ValueError(f'Unknown model kind: {kind}')

    return log_wrap(base) if use_log else base


MODEL_BANK = {name: build_model_from_spec(spec) for name, spec in MODEL_SPECS.items()}

TARGET_SWEEP = {
    'Total Alkalinity': [
        ('C', 'RF_n600_raw'),
        ('C', 'RF_n600_Log'),
        ('C', 'XGB_d4_lr003_Log'),
        ('C', 'Ridge_a30_Log'),
    ],
    'Electrical Conductance': [
        ('C', 'RF_n600_raw'),
        ('C', 'RF_n600_Log'),
        ('C', 'XGB_d4_lr003_Log'),
        ('C', 'Elastic_a0.001_l07_Log'),
    ],
    'Dissolved Reactive Phosphorus': [
        ('C', 'RF_n600_raw'),
        ('C', 'RF_n600_Log'),
        ('C', 'XGB_d3_lr003_Log'),
        ('C', 'Lasso_a0.0010_Log'),
    ],
}


display(pd.DataFrame(
    [(t, fs, m) for t, recipes in TARGET_SWEEP.items() for fs, m in recipes],
    columns=['target', 'feature_set', 'model']
))

,target,feature_set,model
0,Total Alkalinity,C,RF_n600_raw
1,Total Alkalinity,C,RF_n600_Log
2,Total Alkalinity,C,XGB_d4_lr003_Log
3,Total Alkalinity,C,Ridge_a30_Log
4,Electrical Conductance,C,RF_n600_raw
5,Electrical Conductance,C,RF_n600_Log
6,Electrical Conductance,C,XGB_d4_lr003_Log
7,Electrical Conductance,C,Elastic_a0.001_l07_Log
8,Dissolved Reactive Phosphorus,C,RF_n600_raw
9,Dissolved Reactive Phosphorus,C,RF_n600_Log


## Grouped evaluation helpers

This cell:
- runs grouped out-of-fold predictions,
- reports worst-region behavior,
- saves final artifacts for finalists.

In [12]:
def grouped_oof_eval(df_local, target, estimator, features_used, allowed_regions=None):
    '''
    Run grouped OOF evaluation with GroupKFold on spatial clusters.

    Returns:
    - full OOF predictions
    - global metrics
    - fold-by-fold table
    - pseudo-validation holdout R2
    '''
    d = df_local.copy()

    if allowed_regions is not None:
        allowed = set(pd.Series(allowed_regions).astype(str).tolist())
        d = d[d['spatial_group'].astype(str).isin(allowed)].copy()

    X = d[features_used].reset_index(drop=True)
    y = d[target].astype(float).reset_index(drop=True)
    groups = d['spatial_group'].astype(str).reset_index(drop=True)

    n_groups = int(groups.nunique())
    if n_groups < 2:
        raise RuntimeError('Need at least 2 spatial groups for grouped CV.')

    n_splits = min(int(CV_N_SPLITS), n_groups)
    gkf = GroupKFold(n_splits=n_splits)

    pred = np.full(len(d), np.nan, dtype=float)
    fold_rows = []

    for fold_id, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
        pipe = Pipeline([
            ('preprocessor', get_preprocessor(features_used)),
            ('model', clone(estimator))
        ])

        X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
        y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

        pipe.fit(X_tr, y_tr)
        fold_pred = np.asarray(pipe.predict(X_te), dtype=float)
        pred[test_idx] = fold_pred

        fold_rows.append({
            'fold': fold_id,
            'n': int(len(test_idx)),
            'r2': float(r2_score(y_te, fold_pred)),
            'rmse': float(np.sqrt(mean_squared_error(y_te, fold_pred))),
            'mae': float(mean_absolute_error(y_te, fold_pred)),
        })

    if np.isnan(pred).any():
        raise RuntimeError('OOF predictions contain NaN values.')

    fold_df = pd.DataFrame(fold_rows)

    holdout_r2 = np.nan
    if 'is_pseudo_valid' in d.columns:
        hold_mask = d['is_pseudo_valid'].astype(bool).reset_index(drop=True)

        if hold_mask.any() and int((~hold_mask).sum()) >= 2 and int(hold_mask.sum()) >= 2:
            hold_train_groups = set(groups.loc[~hold_mask].tolist())
            hold_test_groups = set(groups.loc[hold_mask].tolist())

            if hold_train_groups.intersection(hold_test_groups):
                raise RuntimeError('Pseudo-holdout leakage detected: train and holdout share groups.')

            hold_pipe = Pipeline([
                ('preprocessor', get_preprocessor(features_used)),
                ('model', clone(estimator))
            ])

            hold_pipe.fit(X.loc[~hold_mask], y.loc[~hold_mask])
            hold_pred = np.asarray(hold_pipe.predict(X.loc[hold_mask]), dtype=float)
            holdout_r2 = float(r2_score(y.loc[hold_mask], hold_pred))

    return {
        'pred': pred,
        'rmse': float(np.sqrt(mean_squared_error(y, pred))),
        'mae': float(mean_absolute_error(y, pred)),
        'r2': float(r2_score(y, pred)),
        'mean_fold_r2': float(fold_df['r2'].mean()) if not fold_df.empty else np.nan,
        'min_fold_r2': float(fold_df['r2'].min()) if not fold_df.empty else np.nan,
        'holdout_r2': holdout_r2,
        'fold_df': fold_df,
        'n_rows': int(len(d)),
        'n_groups': n_groups,
    }


GLOBAL_R2_WEIGHT = 0.60
HOLDOUT_R2_WEIGHT = 0.25
MIN_FOLD_R2_WEIGHT = 0.15


def compute_selection_score(overall_r2, holdout_r2, min_fold_r2=None):
    '''
    Build a finalist selection score with holdout awareness and fold robustness.
    '''
    holdout_term = 0.0 if pd.isna(holdout_r2) else float(holdout_r2)
    min_fold_term = 0.0 if pd.isna(min_fold_r2) else float(min_fold_r2)

    return float(
        GLOBAL_R2_WEIGHT * float(overall_r2) +
        HOLDOUT_R2_WEIGHT * holdout_term +
        MIN_FOLD_R2_WEIGHT * min_fold_term
    )


def fit_full_and_save(df_local, target, estimator, features_used, run_name):
    '''
    Fit the final full-data pipeline and save preprocessor + model artifacts.
    '''
    X = df_local[features_used]
    y = df_local[target].astype(float)

    pre = get_preprocessor(features_used)
    Xp = pre.fit_transform(X)

    mdl = clone(estimator)
    mdl.fit(Xp, y)

    preproc_path = os.path.join(ARTIFACT_DIR, f'{run_name}__preproc.joblib')
    model_path = os.path.join(ARTIFACT_DIR, f'{run_name}__model.joblib')

    joblib.dump(pre, preproc_path)
    joblib.dump(mdl, model_path)

    return preproc_path, model_path

## Stage 1: Scout run

We first run only a narrow shortlist, optionally prioritizing the hardest regions.

In [13]:
SCOUT_GROUPS = None
print('Scout scope: ALL spatial groups')

rows_scout = []

for target in TARGET_COLS:
    for feature_set_name, model_name in TARGET_SWEEP[target]:
        features = features_for_set(df, feature_set_name)
        estimator = clone(MODEL_BANK[model_name])
        run_name = f'SCOUT__{model_name}__{feature_set_name}__{target_key(target)}'

        print(f'\n--- {run_name} ---')

        with mlflow.start_run(run_name=run_name):
            t0 = time.time()

            out = grouped_oof_eval(
                df_local=df,
                target=target,
                estimator=estimator,
                features_used=features,
                allowed_regions=SCOUT_GROUPS
            )

            dt = time.time() - t0

            mlflow.log_param('stage', 'scout')
            mlflow.log_param('target', target)
            mlflow.log_param('model_name', model_name)
            mlflow.log_param('feature_set_name', feature_set_name)
            mlflow.log_param('split_strategy', SPLIT_STRATEGY)
            mlflow.log_param('group_definition_version', GROUP_DEFINITION_VERSION)
            mlflow.log_param('group_values_hash', compute_group_values_hash(df['spatial_group'].astype(str)))
            mlflow.log_param('feature_set_hash', compute_feature_set_hash(features))
            mlflow.log_param('pipeline_version', PIPELINE_VERSION)
            mlflow.log_param('preprocess_version', PREPROCESS_VERSION)
            mlflow.log_param('n_features_used', len(features))

            mlflow.log_metric('r2', out['r2'])
            mlflow.log_metric('mean_fold_r2', out['mean_fold_r2'])
            mlflow.log_metric('min_fold_r2', out['min_fold_r2'])
            mlflow.log_metric('holdout_r2', out['holdout_r2'])
            mlflow.log_metric('rmse', out['rmse'])
            mlflow.log_metric('mae', out['mae'])
            mlflow.log_metric('cv_time_sec', dt)

            selection_score = compute_selection_score(
                overall_r2=out['r2'],
                holdout_r2=out['holdout_r2'],
                min_fold_r2=out['min_fold_r2']
            )

            rows_scout.append({
                'stage': 'scout',
                'run_name': run_name,
                'target': target,
                'model_name': model_name,
                'feature_set': feature_set_name,
                'features_used_json': json.dumps(features),
                'r2': out['r2'],
                'mean_fold_r2': out['mean_fold_r2'],
                'min_fold_r2': out['min_fold_r2'],
                'holdout_r2': out['holdout_r2'],
                'selection_score': selection_score,
                'rmse': out['rmse'],
                'mae': out['mae'],
                'cv_time_sec': dt,
            })

        print(out['fold_df'])
        print(
            f"OOF R2={out['r2']:.4f} | "
            f"Holdout R2={out['holdout_r2']:.4f} | "
            f"selection_score={selection_score:.4f} | "
            f"min_fold_r2={out['min_fold_r2']:.4f}"
        )

scout_df = pd.DataFrame(rows_scout).sort_values(
    ['target', 'selection_score', 'r2', 'min_fold_r2'],
    ascending=[True, False, False, False]
).reset_index(drop=True)

print('\nScout results:')
display(scout_df)

Scout scope: ALL spatial groups

--- SCOUT__RF_n600_raw__C__TotalAlkalinity ---
   fold     n        r2       rmse        mae
0     1  1862 -0.214643  81.730335  63.623316
1     2  1855 -0.117466  80.715082  62.321277
2     3  1844 -0.588027  72.422624  56.196144
3     4  1917 -0.867509  79.234242  67.740310
4     5  1841 -0.653637  74.295982  58.234950
OOF R2=-0.0848 | Holdout R2=0.1810 | selection_score=-0.1357 | min_fold_r2=-0.8675

--- SCOUT__RF_n600_Log__C__TotalAlkalinity ---
   fold     n        r2        rmse        mae
0     1  1862 -0.710908   97.000133  75.498126
1     2  1855 -0.293684   86.846348  66.945476
2     3  1844 -0.591398   72.499466  56.697522
3     4  1917 -0.455923   69.960162  57.997296
4     5  1841 -2.012832  100.284248  83.710697
OOF R2=-0.3292 | Holdout R2=0.0766 | selection_score=-0.4803 | min_fold_r2=-2.0128

--- SCOUT__XGB_d4_lr003_Log__C__TotalAlkalinity ---
   fold     n        r2       rmse        mae
0     1  1862 -0.674987  95.976438  74.812995
1  

,stage,run_name,target,model_name,feature_set,features_used_json,r2,mean_fold_r2,min_fold_r2,holdout_r2,selection_score,rmse,mae,cv_time_sec
0,scout,SCOUT__XGB_d3_lr003_Log__C__DissolvedReactiveP...,Dissolved Reactive Phosphorus,XGB_d3_lr003_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.082391,-0.158870,-0.670999,-0.090545,-0.172721,53.035946,31.943585,1.049699
1,scout,SCOUT__RF_n600_Log__C__DissolvedReactivePhosph...,Dissolved Reactive Phosphorus,RF_n600_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.094195,-0.204447,-0.656414,-0.199442,-0.204840,53.324349,32.280696,15.538017
2,scout,SCOUT__Lasso_a0.0010_Log__C__DissolvedReactive...,Dissolved Reactive Phosphorus,Lasso_a0.0010_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.170085,-0.255772,-0.804567,0.000463,-0.222620,55.142550,34.060417,0.085073
3,scout,SCOUT__RF_n600_raw__C__DissolvedReactivePhosph...,Dissolved Reactive Phosphorus,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.067897,-0.347817,-0.815094,-0.868261,-0.380068,52.679641,37.089449,15.759465
4,scout,SCOUT__Elastic_a0.001_l07_Log__C__ElectricalCo...,Electrical Conductance,Elastic_a0.001_l07_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.041345,-0.196419,-0.650693,0.148936,-0.085177,348.916186,267.374038,0.104364
5,scout,SCOUT__XGB_d4_lr003_Log__C__ElectricalConductance,Electrical Conductance,XGB_d4_lr003_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.094122,-0.232947,-0.434361,0.117129,-0.092345,357.648727,267.079567,1.741644
6,scout,SCOUT__RF_n600_raw__C__ElectricalConductance,Electrical Conductance,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.008935,-0.194391,-0.612216,-0.002008,-0.097696,343.443601,263.838338,13.777503
7,scout,SCOUT__RF_n600_Log__C__ElectricalConductance,Electrical Conductance,RF_n600_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.143655,-0.303106,-0.514784,0.059062,-0.148645,365.654702,272.721617,14.347581
8,scout,SCOUT__RF_n600_raw__C__TotalAlkalinity,Total Alkalinity,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.084777,-0.488256,-0.867509,0.181041,-0.135732,77.790114,61.676895,10.267041
9,scout,SCOUT__Ridge_a30_Log__C__TotalAlkalinity,Total Alkalinity,Ridge_a30_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.190104,-0.558240,-0.944963,0.142155,-0.220268,81.479194,64.986044,0.063878


## Stage 2: Full finalists

For each target, we take the top 2 scout candidates and run full grouped CV.
We also save inference artifacts for those finalists.

In [14]:
def build_finalist_shortlist(scout_df_local, top_n=2):
    '''
    Keep a union of top candidates by complementary views so we do not
    discard globally stronger or more robust models too early.
    '''
    pieces = []

    for target_name, tdf in scout_df_local.groupby('target'):
        by_selection = tdf.sort_values(
            ['selection_score', 'r2', 'min_fold_r2'],
            ascending=[False, False, False]
        ).head(top_n)

        by_global = tdf.sort_values(
            ['r2', 'min_fold_r2', 'selection_score'],
            ascending=[False, False, False]
        ).head(top_n)

        by_robust = tdf.sort_values(
            ['min_fold_r2', 'r2', 'selection_score'],
            ascending=[False, False, False]
        ).head(1)

        shortlist = pd.concat([by_selection, by_global, by_robust], axis=0)
        shortlist = shortlist.drop_duplicates(subset=['run_name']).reset_index(drop=True)
        pieces.append(shortlist)

    return pd.concat(pieces, axis=0).reset_index(drop=True)


finalist_df = build_finalist_shortlist(scout_df, top_n=2)

print('Finalists (union shortlist):')
display(finalist_df[[
    'target',
    'feature_set',
    'model_name',
    'selection_score',
    'holdout_r2',
    'r2',
    'min_fold_r2',
    'run_name'
]])

rows_full = []
OOF_PREDS = {}

for _, row in finalist_df.iterrows():
    target = row['target']
    model_name = row['model_name']
    feature_set_name = row['feature_set']
    features = json.loads(row['features_used_json'])
    estimator = clone(MODEL_BANK[model_name])

    run_name = f'FULL__{model_name}__{feature_set_name}__{target_key(target)}'
    print(f'\n=== {run_name} ===')

    with mlflow.start_run(run_name=run_name):
        t0 = time.time()

        out = grouped_oof_eval(
            df_local=df,
            target=target,
            estimator=estimator,
            features_used=features,
            allowed_regions=None
        )

        dt = time.time() - t0

        preproc_path, model_path = fit_full_and_save(
            df_local=df,
            target=target,
            estimator=estimator,
            features_used=features,
            run_name=run_name
        )

        mlflow.log_param('stage', 'full')
        mlflow.log_param('target', target)
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('feature_set_name', feature_set_name)
        mlflow.log_param('split_strategy', SPLIT_STRATEGY)
        mlflow.log_param('group_definition_version', GROUP_DEFINITION_VERSION)
        mlflow.log_param('group_values_hash', compute_group_values_hash(df['spatial_group'].astype(str)))
        mlflow.log_param('feature_set_hash', compute_feature_set_hash(features))
        mlflow.log_param('pipeline_version', PIPELINE_VERSION)
        mlflow.log_param('preprocess_version', PREPROCESS_VERSION)
        mlflow.log_param('n_features_used', len(features))

        mlflow.log_metric('r2', out['r2'])
        mlflow.log_metric('mean_fold_r2', out['mean_fold_r2'])
        mlflow.log_metric('min_fold_r2', out['min_fold_r2'])
        mlflow.log_metric('holdout_r2', out['holdout_r2'])
        mlflow.log_metric('rmse', out['rmse'])
        mlflow.log_metric('mae', out['mae'])
        mlflow.log_metric('cv_time_sec', dt)

        mlflow.log_artifact(preproc_path, artifact_path='submission_assets')
        mlflow.log_artifact(model_path, artifact_path='submission_assets')

        OOF_PREDS[run_name] = out['pred']

        selection_score = compute_selection_score(
            overall_r2=out['r2'],
            holdout_r2=out['holdout_r2'],
            min_fold_r2=out['min_fold_r2']
        )

        rows_full.append({
            'stage': 'full',
            'run_name': run_name,
            'target': target,
            'model_name': model_name,
            'feature_set': feature_set_name,
            'features_used_json': json.dumps(features),
            'r2': out['r2'],
            'mean_fold_r2': out['mean_fold_r2'],
            'min_fold_r2': out['min_fold_r2'],
            'holdout_r2': out['holdout_r2'],
            'selection_score': selection_score,
            'rmse': out['rmse'],
            'mae': out['mae'],
            'cv_time_sec': dt,
            'preproc_path': preproc_path,
            'model_path': model_path,
        })

    print(out['fold_df'])
    print(
        f"FULL OOF R2={out['r2']:.4f} | "
        f"Holdout R2={out['holdout_r2']:.4f} | "
        f"selection_score={selection_score:.4f} | "
        f"min_fold_r2={out['min_fold_r2']:.4f}"
    )

full_df = pd.DataFrame(rows_full).sort_values(
    ['target', 'selection_score', 'r2', 'min_fold_r2'],
    ascending=[True, False, False, False]
).reset_index(drop=True)

print('\nFull results:')
display(full_df)

Finalists (union shortlist):


,target,feature_set,model_name,selection_score,holdout_r2,r2,min_fold_r2,run_name
0,Dissolved Reactive Phosphorus,C,XGB_d3_lr003_Log,-0.172721,-0.090545,-0.082391,-0.670999,SCOUT__XGB_d3_lr003_Log__C__DissolvedReactiveP...
1,Dissolved Reactive Phosphorus,C,RF_n600_Log,-0.204840,-0.199442,-0.094195,-0.656414,SCOUT__RF_n600_Log__C__DissolvedReactivePhosph...
2,Dissolved Reactive Phosphorus,C,RF_n600_raw,-0.380068,-0.868261,-0.067897,-0.815094,SCOUT__RF_n600_raw__C__DissolvedReactivePhosph...
3,Electrical Conductance,C,Elastic_a0.001_l07_Log,-0.085177,0.148936,-0.041345,-0.650693,SCOUT__Elastic_a0.001_l07_Log__C__ElectricalCo...
4,Electrical Conductance,C,XGB_d4_lr003_Log,-0.092345,0.117129,-0.094122,-0.434361,SCOUT__XGB_d4_lr003_Log__C__ElectricalConductance
5,Electrical Conductance,C,RF_n600_raw,-0.097696,-0.002008,-0.008935,-0.612216,SCOUT__RF_n600_raw__C__ElectricalConductance
6,Total Alkalinity,C,RF_n600_raw,-0.135732,0.181041,-0.084777,-0.867509,SCOUT__RF_n600_raw__C__TotalAlkalinity
7,Total Alkalinity,C,Ridge_a30_Log,-0.220268,0.142155,-0.190104,-0.944963,SCOUT__Ridge_a30_Log__C__TotalAlkalinity



=== FULL__XGB_d3_lr003_Log__C__DissolvedReactivePhosphorus ===
   fold     n        r2       rmse        mae
0     1  1862 -0.670999  83.212232  61.678584
1     2  1855 -0.073307  57.361007  38.850563
2     3  1844 -0.083347  46.987545  25.137075
3     4  1917  0.011329  25.979827  16.217600
4     5  1841  0.021975  31.708474  18.102685
FULL OOF R2=-0.0824 | Holdout R2=-0.0905 | selection_score=-0.1727 | min_fold_r2=-0.6710

=== FULL__RF_n600_Log__C__DissolvedReactivePhosphorus ===
   fold     n        r2       rmse        mae
0     1  1862 -0.656414  82.848289  61.542996
1     2  1855 -0.067828  57.214412  38.782327
2     3  1844 -0.106525  47.487536  25.342381
3     4  1917 -0.130752  27.783952  17.004856
4     5  1841 -0.060717  33.021743  18.989611
FULL OOF R2=-0.0942 | Holdout R2=-0.1994 | selection_score=-0.2048 | min_fold_r2=-0.6564

=== FULL__RF_n600_raw__C__DissolvedReactivePhosphorus ===
   fold     n        r2       rmse        mae
0     1  1862 -0.422190  76.767574  59.846

,stage,run_name,target,model_name,feature_set,features_used_json,r2,mean_fold_r2,min_fold_r2,holdout_r2,selection_score,rmse,mae,cv_time_sec,preproc_path,model_path
0,full,FULL__XGB_d3_lr003_Log__C__DissolvedReactivePh...,Dissolved Reactive Phosphorus,XGB_d3_lr003_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.082391,-0.158870,-0.670999,-0.090545,-0.172721,53.035946,31.943585,0.741817,../models/final_deadline_mvp4\FULL__XGB_d3_lr0...,../models/final_deadline_mvp4\FULL__XGB_d3_lr0...
1,full,FULL__RF_n600_Log__C__DissolvedReactivePhosphorus,Dissolved Reactive Phosphorus,RF_n600_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.094195,-0.204447,-0.656414,-0.199442,-0.204840,53.324349,32.280696,13.346480,../models/final_deadline_mvp4\FULL__RF_n600_Lo...,../models/final_deadline_mvp4\FULL__RF_n600_Lo...
2,full,FULL__RF_n600_raw__C__DissolvedReactivePhosphorus,Dissolved Reactive Phosphorus,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.067897,-0.347817,-0.815094,-0.868261,-0.380068,52.679641,37.089449,17.054619,../models/final_deadline_mvp4\FULL__RF_n600_ra...,../models/final_deadline_mvp4\FULL__RF_n600_ra...
3,full,FULL__Elastic_a0.001_l07_Log__C__ElectricalCon...,Electrical Conductance,Elastic_a0.001_l07_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.041345,-0.196419,-0.650693,0.148936,-0.085177,348.916186,267.374038,0.094221,../models/final_deadline_mvp4\FULL__Elastic_a0...,../models/final_deadline_mvp4\FULL__Elastic_a0...
4,full,FULL__XGB_d4_lr003_Log__C__ElectricalConductance,Electrical Conductance,XGB_d4_lr003_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.094122,-0.232947,-0.434361,0.117129,-0.092345,357.648727,267.079567,1.633867,../models/final_deadline_mvp4\FULL__XGB_d4_lr0...,../models/final_deadline_mvp4\FULL__XGB_d4_lr0...
5,full,FULL__RF_n600_raw__C__ElectricalConductance,Electrical Conductance,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.008935,-0.194391,-0.612216,-0.002008,-0.097696,343.443601,263.838338,15.562163,../models/final_deadline_mvp4\FULL__RF_n600_ra...,../models/final_deadline_mvp4\FULL__RF_n600_ra...
6,full,FULL__RF_n600_raw__C__TotalAlkalinity,Total Alkalinity,RF_n600_raw,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.084777,-0.488256,-0.867509,0.181041,-0.135732,77.790114,61.676895,16.391377,../models/final_deadline_mvp4\FULL__RF_n600_ra...,../models/final_deadline_mvp4\FULL__RF_n600_ra...
7,full,FULL__Ridge_a30_Log__C__TotalAlkalinity,Total Alkalinity,Ridge_a30_Log,C,"[""swir22"", ""NDMI"", ""MNDWI"", ""pet""]",-0.190104,-0.558240,-0.944963,0.142155,-0.220268,81.479194,64.986044,0.131424,../models/final_deadline_mvp4\FULL__Ridge_a30_...,../models/final_deadline_mvp4\FULL__Ridge_a30_...


## Freeze Manifest A

This chooses one safe anchor model per target from the full finalists.
For DRP, we prefer the safer linear models.

In [74]:
TARGET_GLOBAL_R2_FLOOR = {
    'Total Alkalinity': 0.00,
    'Electrical Conductance': 0.00,
    'Dissolved Reactive Phosphorus': -0.20,
}

DRP_SAFE_MODELS = [
    'XGB_d3_lr003_Log',
    'Lasso_a0.0010_Log',
    'Ridge_a30_Log',
    'RF_n600_raw',
    'RF_n600_Log',
]

In [ ]:
manifest_A = {}
manifest_B = {}

for target in TARGET_COLS:
    tdf = full_df[full_df['target'] == target].copy()

    # sort by geography-aware score first
    tdf = tdf.sort_values(
        ['selection_score', 'r2', 'min_fold_r2'],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    floor = TARGET_GLOBAL_R2_FLOOR[target]
    passed_floor = tdf[tdf['r2'] >= floor].copy()

    if target == 'Total Alkalinity':
        chosen_A = passed_floor.iloc[0] if not passed_floor.empty else tdf.iloc[0]
        chosen_B = chosen_A

    elif target == 'Electrical Conductance':
        safe_pool = passed_floor.copy()

        if safe_pool.empty:
            safe_pool = tdf.sort_values(['r2', 'selection_score'], ascending=[False, False])

        chosen_A = safe_pool.iloc[0]
        challenger_pool = tdf.copy()
        chosen_B = challenger_pool.iloc[0]

    elif target == 'Dissolved Reactive Phosphorus':
        safe_pool = tdf[tdf['model_name'].isin(DRP_SAFE_MODELS)].copy()

        if safe_pool.empty:
            safe_pool = tdf.copy()

        safe_pool_A = safe_pool.sort_values(
            ['r2', 'min_fold_r2', 'selection_score'],
            ascending=[False, False, False]
        )
        chosen_A = safe_pool_A.iloc[0]

        safe_pool_B = safe_pool.sort_values(
            ['selection_score', 'holdout_r2', 'r2'],
            ascending=[False, False, False]
        )
        chosen_B = safe_pool_B.iloc[0]

    manifest_A[target] = chosen_A.to_dict()
    manifest_B[target] = chosen_B.to_dict()

print('=== MANIFEST A (SAFE) ===')
print(json.dumps({
    k: {
        'run_name': v['run_name'],
        'model_name': v['model_name'],
        'feature_set': v['feature_set'],
        'selection_score': float(v['selection_score']),
        'holdout_r2': float(v['holdout_r2']) if pd.notna(v['holdout_r2']) else None,
        'r2': float(v['r2']),
        'min_fold_r2': float(v['min_fold_r2']),
    }
    for k, v in manifest_A.items()
}, indent=2))

print('\n=== MANIFEST B (CHALLENGER) ===')
print(json.dumps({
    k: {
        'run_name': v['run_name'],
        'model_name': v['model_name'],
        'feature_set': v['feature_set'],
        'selection_score': float(v['selection_score']),
        'holdout_r2': float(v['holdout_r2']) if pd.notna(v['holdout_r2']) else None,
        'r2': float(v['r2']),
        'min_fold_r2': float(v['min_fold_r2']),
    }
    for k, v in manifest_B.items()
}, indent=2))

## Diagnostic For Retraining Before Making Submission

In [76]:
cols = ['swir22', 'NDMI', 'MNDWI', 'pet'] + TARGET_COLS
display(df[cols].describe(percentiles=[0.5, 0.9, 0.95, 0.99, 0.999]).T)

,count,mean,std,min,50%,90%,95%,99%,99.9%,max
basin_upstream_area_km2,9319.0,24924.741120,87167.528604,1.335000e+02,4688.700000,47423.000000,80358.300000,764462.400000,786079.800000,786079.800000
worldpop_mean_1km,9319.0,2.664821,3.879033,0.000000e+00,1.199995,8.020488,10.694789,18.795643,26.793154,26.793154
river_avg_discharge_cms,9319.0,26.927536,55.863831,1.100000e-02,6.446000,75.058998,118.915001,363.572998,436.578003,436.578003
pop_density_upstream,9319.0,0.001827,0.005071,0.000000e+00,0.000256,0.003671,0.010484,0.026349,0.058574,0.058574
specific_discharge,9319.0,0.002568,0.002892,7.648720e-07,0.001271,0.006732,0.009292,0.010869,0.013457,0.013457


## Submission helper

This loads the saved artifacts for a chosen manifest entry and generates clipped predictions.

In [77]:
def predict_from_manifest_entry(entry, df_val_local):
    '''
    Predict from a frozen manifest entry using the saved
    preprocessor and model artifacts.
    '''
    feats = json.loads(entry['features_used_json'])
    pre = joblib.load(entry['preproc_path'])
    mdl = joblib.load(entry['model_path'])

    X = df_val_local[feats]
    pred = mdl.predict(pre.transform(X))
    pred = np.asarray(pred, dtype=float)

    return np.clip(pred, 0, None)

## Build Shot A / B / C

- **Shot A** = safe anchor from Manifest A
- **Shot B** = EC aggressive + DRP safe shrink
- **Shot C** = hedge blend between A and B

In [78]:
df_val = pd.read_parquet('../data/interim/water_quality_mvp_validation.parquet')
df_val = engineer_features(df_val)

tpl = pd.read_csv('../data/raw/submission_template.csv')
tpl = make_row_id_template(tpl)

# Validate that all needed features exist in validation
needed_feats = set()
for t in TARGET_COLS:
    needed_feats.update(json.loads(manifest_A[t]['features_used_json']))
    needed_feats.update(json.loads(manifest_B[t]['features_used_json']))

missing_feats = sorted([f for f in needed_feats if f not in df_val.columns])
if missing_feats:
    raise RuntimeError(f'Missing validation features: {missing_feats}')

def clip_by_train_quantile(pred, target, q_hi=0.995):
    hi = float(df[target].quantile(q_hi))
    return np.clip(np.asarray(pred, dtype=float), 0, hi)

# -------------------------
# Shot A: safe anchor
# -------------------------
shotA = tpl.copy()
for target in TARGET_COLS:
    pred_a = predict_from_manifest_entry(manifest_A[target], df_val)
    shotA[target] = clip_by_train_quantile(pred_a, target)

assert_submission_integrity(shotA, tpl, TARGET_COLS)

# -------------------------
# Shot B: challenger blend
# -------------------------
shotB = tpl.copy()

# TA stays mostly safe
shotB['Total Alkalinity'] = shotA['Total Alkalinity']

# EC: blend safe + challenger to reduce catastrophic swaps
if manifest_B['Electrical Conductance']['run_name'] != manifest_A['Electrical Conductance']['run_name']:
    ec_safe = shotA['Electrical Conductance'].values
    ec_chal = predict_from_manifest_entry(manifest_B['Electrical Conductance'], df_val)
    ec_blend = 0.35 * ec_safe + 0.65 * ec_chal
    shotB['Electrical Conductance'] = clip_by_train_quantile(ec_blend, 'Electrical Conductance')
    print('Shot B EC challenger blend:', manifest_B['Electrical Conductance']['run_name'])
else:
    shotB['Electrical Conductance'] = shotA['Electrical Conductance']
    print('Shot B EC kept from Manifest A')

# DRP: challenger + median shrink
drp_chal = predict_from_manifest_entry(manifest_B['Dissolved Reactive Phosphorus'], df_val)
drp_train_median = float(df['Dissolved Reactive Phosphorus'].median())
drp_blend = 0.55 * drp_chal + 0.45 * drp_train_median
shotB['Dissolved Reactive Phosphorus'] = clip_by_train_quantile(
    drp_blend, 'Dissolved Reactive Phosphorus'
)
print('Shot B DRP challenger:', manifest_B['Dissolved Reactive Phosphorus']['run_name'])

assert_submission_integrity(shotB, tpl, TARGET_COLS)

# -------------------------
# Shot C: hedge (middle risk)
# -------------------------
shotC = tpl.copy()
shotC['Total Alkalinity'] = clip_by_train_quantile(
    0.85 * shotA['Total Alkalinity'] + 0.15 * shotB['Total Alkalinity'],
    'Total Alkalinity'
)
shotC['Electrical Conductance'] = clip_by_train_quantile(
    0.50 * shotA['Electrical Conductance'] + 0.50 * shotB['Electrical Conductance'],
    'Electrical Conductance'
)
shotC['Dissolved Reactive Phosphorus'] = clip_by_train_quantile(
    0.45 * shotA['Dissolved Reactive Phosphorus'] + 0.55 * shotB['Dissolved Reactive Phosphorus'],
    'Dissolved Reactive Phosphorus'
)

assert_submission_integrity(shotC, tpl, TARGET_COLS)

# -------------------------
# Save
# -------------------------
stamp = datetime.now().strftime('%Y%m%d_%H%M')
pathA = f'../data/submission/submission_{stamp}_A_safe.csv'
pathB = f'../data/submission/submission_{stamp}_B_challenger_blend.csv'
pathC = f'../data/submission/submission_{stamp}_C_hedge.csv'

os.makedirs('../data/submission', exist_ok=True)

shotA.drop(columns=['row_id']).to_csv(pathA, index=False)
shotB.drop(columns=['row_id']).to_csv(pathB, index=False)
shotC.drop(columns=['row_id']).to_csv(pathC, index=False)

print('Saved files:')
print('A:', pathA)
print('B:', pathB)
print('C:', pathC)

Shot B EC kept from Manifest A
Shot B DRP challenger: FULL__XGB_d3_lr003_Log__B__DissolvedReactivePhosphorus
Saved files:
A: ../data/submission/submission_20260312_1832_A_safe.csv
B: ../data/submission/submission_20260312_1832_B_challenger_blend.csv
C: ../data/submission/submission_20260312_1832_C_hedge.csv


## Submission diagnostics

This final cell prints simple distribution summaries for the three submission variants.

In [79]:
def summarize_shot(shot_df, name):
    print(f'\n{name} stats')
    stats = shot_df[TARGET_COLS].describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
    ).T
    display(stats[['min', '1%', '5%', '50%', 'mean', '95%', '99%', 'max']])

summarize_shot(shotA, 'Shot A')
summarize_shot(shotB, 'Shot B')
summarize_shot(shotC, 'Shot C')

print('\nMean absolute deltas vs Shot A')
delta_tbl = pd.DataFrame({
    'target': TARGET_COLS,
    'B_vs_A_mae': [float(np.mean(np.abs(shotB[t] - shotA[t]))) for t in TARGET_COLS],
    'C_vs_A_mae': [float(np.mean(np.abs(shotC[t] - shotA[t]))) for t in TARGET_COLS],
})
display(delta_tbl)

tracker = pd.DataFrame([
    {'file': pathA, 'hypothesis': 'Safety anchor (lowest variance)'},
    {'file': pathC, 'hypothesis': 'Balanced hedge between A and B'},
    {'file': pathB, 'hypothesis': 'Most aggressive on EC/DRP challenger blend'},
])

print('\nSubmission tracker:')
display(tracker)

print('\nSuggested upload order: A -> C -> B')


Shot A stats


,min,1%,5%,50%,mean,95%,99%,max
Total Alkalinity,13.198025,26.015306,29.328127,60.725183,66.072442,146.109953,169.846849,180.237457
Electrical Conductance,105.549446,112.207654,117.714803,279.766144,312.345208,595.956067,711.255640,793.762146
Dissolved Reactive Phosphorus,11.914917,12.133108,12.759013,19.470349,22.866611,34.635423,57.549519,58.136471



Shot B stats


,min,1%,5%,50%,mean,95%,99%,max
Total Alkalinity,13.198025,26.015306,29.328127,60.725183,66.072442,146.109953,169.846849,180.237457
Electrical Conductance,105.549446,112.207654,117.714803,279.766144,312.345208,595.956067,711.255640,793.762146
Dissolved Reactive Phosphorus,15.553204,15.673210,16.017457,19.708692,21.576636,28.049483,40.652235,40.975059



Shot C stats


...


Mean absolute deltas vs Shot A


,target,B_vs_A_mae,C_vs_A_mae
0,Total Alkalinity,0.000000,1.438849e-15
1,Electrical Conductance,0.000000,0.000000e+00
2,Dissolved Reactive Phosphorus,3.213897,1.767643e+00



Submission tracker:


,file,hypothesis
0,../data/submission/submission_20260312_1832_A_...,Safety anchor (lowest variance)
1,../data/submission/submission_20260312_1832_C_...,Balanced hedge between A and B
2,../data/submission/submission_20260312_1832_B_...,Most aggressive on EC/DRP challenger blend



Suggested upload order: A -> C -> B
